# From-Scratch MoE GPT — Free Colab T4 Pretrain (multi-session, auto-resume)

This notebook pretrains a **small Mixture-of-Experts GPT from random init**
(~202.9M total / ~69.5M active params) on open code+text corpora, across
multiple free Colab T4 sessions.

**SAFE TO RE-RUN EVERY SESSION.** Kill it at the ~12h cap, reconnect, and
`Runtime → Run all` again — it auto-resumes from the latest Google Drive
checkpoint (exact step / token-count / AdamW state / RNG / LR schedule).

> ### HONEST scope (read this)
> A single T4 pretrains a small MoE that learns **code SYNTAX and simple
> completions** — it is **NOT a real coding assistant**, has no instruction
> tuning, and will not implement functions or reason about bugs. Frontier /
> 8B-active scale needs a multi-GPU cluster + trillions of tokens (see the last
> cell). The "8" here means **8 experts**, never 8 billion params.

**Steps below:** (1) pick a T4 runtime, (2) mount Drive, (3) get the code,
(4) install deps, (5) optional HF/AWS auth, (6) prepare data (resumable),
(7) train (auto-resume). Re-run the whole notebook each session.

## 0. Confirm a GPU runtime

`Runtime → Change runtime type → Hardware accelerator: T4 GPU`, then run this.
If it prints `cuda? False`, fix the runtime before continuing — CPU training is
not viable for the 6B-token budget.

In [ ]:
import torch, subprocess, sys
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda?  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu    :', torch.cuda.get_device_name(0))
    try:
        print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode().strip())
    except Exception as e:
        print('nvidia-smi unavailable:', e)
else:
    print('!! No CUDA. Runtime -> Change runtime type -> T4 GPU, then re-run.')

## 1. Mount Google Drive (persistence across sessions)

Everything that must survive a disconnect — packed data, tokenizer, and the
resume checkpoints — lives under `MyDrive/moe_pretrain`. Approve the auth popup.
Re-running this cell when Drive is already mounted is a no-op.

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/moe_pretrain'
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as e:
    # not on Colab (e.g. local Jupyter): fall back to a local persistent dir
    print('[drive] not on Colab (', e, ') -> using a local DRIVE_ROOT instead.')
    DRIVE_ROOT = os.path.abspath('./moe_pretrain_run')

DATA_DIR = os.path.join(DRIVE_ROOT, 'data')
CKPT_DIR = os.path.join(DRIVE_ROOT, 'ckpts')
CODE_DIR = os.path.join(DRIVE_ROOT, 'pretrain_moe')   # code copied here so it persists
CONFIG   = os.path.join(DRIVE_ROOT, 'winner_config.json')
for d in (DATA_DIR, CKPT_DIR, CODE_DIR):
    os.makedirs(d, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('DATA_DIR   =', DATA_DIR)
print('CKPT_DIR   =', CKPT_DIR)
print('CODE_DIR   =', CODE_DIR)

## 2. Get the code onto Drive (model.py / prepare_data.py / train.py)

Pick ONE source below by editing `CODE_SOURCE`:
- `'github'` — clone/pull a repo that contains `pretrain_moe/` (set `REPO_URL`).
- `'upload'` — you already uploaded the three `.py` files to `CODE_DIR` (or to
  `/content`); this cell just verifies/copies them into `CODE_DIR`.

Copying into `CODE_DIR` (on Drive) means the code itself survives reconnects, so
later sessions don't depend on re-cloning.

In [ ]:
import os, shutil, subprocess

CODE_SOURCE = 'github'          # 'github' | 'upload'
REPO_URL    = 'https://github.com/YDN/weaver.git'   # <-- EDIT to your repo
REPO_SUBDIR = 'windsurf-project/pretrain_moe'        # path to pretrain_moe in the repo

NEEDED = ('model.py', 'prepare_data.py', 'train.py')

def _have_all(d):
    return all(os.path.exists(os.path.join(d, f)) for f in NEEDED)

if CODE_SOURCE == 'github':
    clone_dir = '/content/_pretrain_repo'
    if os.path.isdir(os.path.join(clone_dir, '.git')):
        subprocess.run(['git', '-C', clone_dir, 'pull', '--ff-only'], check=False)
    else:
        shutil.rmtree(clone_dir, ignore_errors=True)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, clone_dir], check=True)
    src = os.path.join(clone_dir, REPO_SUBDIR)
    for f in NEEDED:
        shutil.copy2(os.path.join(src, f), os.path.join(CODE_DIR, f))
elif CODE_SOURCE == 'upload':
    # accept files already in CODE_DIR, else pull them from /content
    if not _have_all(CODE_DIR):
        for f in NEEDED:
            cand = os.path.join('/content', f)
            if os.path.exists(cand):
                shutil.copy2(cand, os.path.join(CODE_DIR, f))
else:
    raise ValueError('CODE_SOURCE must be github or upload')

assert _have_all(CODE_DIR), f'Missing one of {NEEDED} in {CODE_DIR}. Fix CODE_SOURCE/REPO_URL.'
print('code ready in', CODE_DIR, '->', os.listdir(CODE_DIR))

## 3. Install dependencies

Colab ships torch + numpy; we add the data-pipeline deps. Idempotent — safe to
re-run. (`smart_open`/`boto3` are only used if you wire up The-Stack-v2 S3.)

In [ ]:
import sys, subprocess
pkgs = ['datasets', 'tokenizers', 'huggingface_hub', 'smart_open[s3]', 'boto3']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *pkgs], check=False)
# torch/numpy are preinstalled on Colab; only install if somehow missing.
try:
    import torch, numpy  # noqa: F401
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy'], check=False)
print('deps installed.')

## 4. (Optional) auth for gated / pointer shards

Both are **optional** — the pipeline skips them cleanly if absent:
- **HuggingFace login** unlocks the gated `starcoderdata` shard (accept its ToS
  on the HF model page first).
- **AWS creds** let The-Stack-v2 fetch real file content from Software Heritage
  S3 (its rows are `blob_id` pointers, not code). Without creds that shard is
  skipped and the other three carry the mix.

Leave the placeholders empty to skip.

In [ ]:
import os
HF_TOKEN = ''   # <-- optional: a HuggingFace token with starcoderdata ToS accepted
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('HuggingFace: logged in.')
else:
    print('HuggingFace: no token -> gated starcoderdata shard will be skipped.')

AWS_KEY = ''    # <-- optional: AWS_ACCESS_KEY_ID for The-Stack-v2 S3 content fetch
AWS_SECRET = '' # <-- optional: AWS_SECRET_ACCESS_KEY
if AWS_KEY and AWS_SECRET:
    os.environ['AWS_ACCESS_KEY_ID'] = AWS_KEY
    os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
    print('AWS: creds set -> The-Stack-v2 content fetch enabled.')
else:
    print('AWS: no creds -> The-Stack-v2 (pointer rows) skipped; other 3 shards carry the mix.')

## 5. Write the winner config to Drive (idempotent)

The architecture JSON. `train.py` also defaults to these exact values, but
writing it to Drive pins the config hash logged in every resume bundle.

In [ ]:
import json, os
WINNER = {
    'vocab_size': 16384, 'ctx_len': 1024,
    'n_layer': 9, 'n_head': 10, 'n_embd': 640, 'head_dim': 64,
    'n_experts': 8, 'top_k': 2, 'd_ff': 1920,
    'capacity_factor': 1.25, 'drop_overflow_tokens': True,
    'router_noise_std': 1.0, 'aux_loss_coef': 0.01, 'router_z_loss_coef': 0.001,
    'use_qk_norm': True, 'dropout': 0.0, 'bias': False, 'init_std': 0.02,
}
if not os.path.exists(CONFIG):
    with open(CONFIG, 'w') as f:
        json.dump(WINNER, f, indent=2)
    print('wrote', CONFIG)
else:
    print('config already on Drive ->', CONFIG)

# Quick architecture self-test (param accounting + a fwd/bwd on random tokens).
import subprocess, sys
subprocess.run([sys.executable, os.path.join(CODE_DIR, 'model.py')], check=False)

## 6. Prepare data — train 16k BPE + pack train.bin/val.bin (RESUMABLE)

Streams the 4 shards, fits the byte-level BPE on a sampled slice, then packs
EOS-delimited tokens into `data/train.bin` + `data/val.bin` on Drive. A
per-substream cursor (`data/cursor.json`, atomic) makes this **resume** where it
stopped — re-running continues packing, never restarts.

You do **not** need the full 6B packed before training: training samples random
windows from whatever is already packed, and you can re-run this cell in later
sessions to extend the corpus.

In [ ]:
import subprocess, sys, os
TOKEN_BUDGET_B = 6.0
ret = subprocess.run([
    sys.executable, os.path.join(CODE_DIR, 'prepare_data.py'),
    '--data-dir', DATA_DIR,
    '--token-budget-b', str(TOKEN_BUDGET_B),
], check=False)
print('prepare_data.py exit code:', ret.returncode)
meta = os.path.join(DATA_DIR, 'meta.json')
if os.path.exists(meta):
    import json; print('meta:', json.load(open(meta)))

## 7. Train — multi-session loop, AUTO-RESUMES from Drive

This is the long-running cell. It auto-loads the latest valid checkpoint from
`ckpts/`, restores step/tokens/optimizer/RNG/schedule exactly, and trains until
Colab kills the session at the ~12h cap. A resume bundle is written atomically
every `ckpt_interval` steps, so a kill at any moment loses at most a few steps.

**When the session dies: reconnect → `Runtime → Run all` → this cell resumes.**
Repeat ~9 times until the 6B budget is met (the loop no-ops once it's reached).

In [ ]:
import subprocess, sys, os
cmd = [
    sys.executable, os.path.join(CODE_DIR, 'train.py'),
    '--data-dir', DATA_DIR,
    '--ckpt-dir', CKPT_DIR,
    '--config',   CONFIG,
    '--token-budget-b', str(6.0),
]
print('launching:', ' '.join(cmd))
# stream the trainer's stdout live into the notebook
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
finally:
    proc.wait()
print('train.py exit code:', proc.returncode)

## 8. (Optional) peek at the latest checkpoint

Confirms the resume bundle on Drive — step, token count, and that the full AdamW
state is present (so the next session truly continues, not restarts).

In [ ]:
import torch, os, glob
latest = os.path.join(CKPT_DIR, 'ckpt_latest.pt')
if not os.path.exists(latest):
    cands = sorted(glob.glob(os.path.join(CKPT_DIR, 'ckpt_step*.pt')))
    latest = cands[-1] if cands else None
if latest:
    b = torch.load(latest, map_location='cpu', weights_only=False)
    print('checkpoint  :', os.path.basename(latest))
    print('global_step :', b.get('global_step'))
    print('token_count :', f"{b.get('token_count',0)/1e6:.1f}M")
    print('best_val    :', b.get('best_val'))
    print('has AdamW m/v:', bool(b.get('optimizer',{}).get('state')))
    print('arch hash   :', b.get('model_config_hash'))
else:
    print('no checkpoint yet — run cell 7 long enough to hit ckpt_interval.')

---
## 8B-ACTIVE scale-up — config-only, but NOT possible on this T4

The **same code** reaches a Mixtral-class **~8B-active / ~47B-total** model with
a **config-only** change — keep 8 experts / top-2, grow the shape:

```python
MoEGPTConfig(
    n_embd=4096, n_layer=32, head_dim=128, n_head=32,
    d_ff=14336, n_experts=8, top_k=2,
    vocab_size=65536, ctx_len=4096,
)   # ~8B active / ~47B total
```

**Why it cannot run here (a hardware wall, not a tuning knob):** all 8 expert
weights **plus their full fp32 AdamW state** must be resident — MoE saves FLOPs,
not memory. At ~47B total params the optimizer state alone is hundreds of GB.
It needs a **multi-GPU cluster** with expert-parallel + tensor-parallel sharding
(Megatron / DeepSpeed-MoE), **ZeRO-3** optimizer-state sharding, and a token
budget in the **trillions** (~15-20x params). Only the config + parallelism
strategy change — never the architecture family, the router stack, or this
resume bundle. This notebook is the small, honest, T4-runnable end of exactly
that spectrum.